<a href="https://colab.research.google.com/github/putricantya96-pixel/gradaspal/blob/main/GradAspal_Colab_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GradAspal di Google Colab
Jalankan sel dari atas ke bawah (Shift+Enter). Sel terakhir menampilkan link aplikasi.

In [1]:
!pip install -q streamlit openpyxl
!npm install -q localtunnel
!mkdir -p .streamlit

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 2s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [2]:
%%writefile app.py
"""GradAspal - Perencanaan Gradasi Agregat & Kadar Aspal Estimasi
Latasir, Lataston, Laston.  Jalankan: streamlit run app.py
"""
import csv
import hashlib
import io
import json
import os
import re
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st

# Saringan spesifikasi (mm) dan batas gradasi % lolos (BB, BA)
S = [37.5, 25, 19, 12.5, 9.5, 4.75, 2.36, 1.18, 0.6, 0.3, 0.15, 0.075]
SPEC = {
    "Latasir Kelas A": {19: (100, 100), 9.5: (90, 100), 0.075: (10, 15)},
    "Latasir Kelas B": {19: (100, 100), 2.36: (75, 100), 0.075: (8, 13)},
    "Lataston WC (Senjang)": {19: (100, 100), 12.5: (90, 100), 9.5: (75, 85), 2.36: (50, 72), 0.6: (35, 60), 0.075: (6, 10)},
    "Lataston Base (Senjang)": {19: (100, 100), 12.5: (90, 100), 9.5: (65, 90), 2.36: (35, 55), 0.6: (15, 35), 0.075: (2, 9)},
    "Lataston WC (Semi Senjang)": {19: (100, 100), 12.5: (87, 100), 9.5: (55, 88), 2.36: (50, 62), 0.6: (20, 45), 0.3: (15, 35), 0.075: (6, 10)},
    "Lataston Base (Semi Senjang)": {19: (100, 100), 12.5: (90, 100), 9.5: (55, 70), 2.36: (32, 44), 0.6: (15, 35), 0.3: (5, 35), 0.075: (4, 8)},
    "Laston AC-WC": {19: (100, 100), 12.5: (90, 100), 9.5: (77, 90), 4.75: (53, 69), 2.36: (33, 53), 1.18: (21, 40), 0.6: (14, 30), 0.3: (9, 22), 0.15: (6, 15), 0.075: (4, 9)},
    "Laston AC-BC": {25: (100, 100), 19: (90, 100), 12.5: (75, 90), 9.5: (66, 82), 4.75: (46, 64), 2.36: (30, 49), 1.18: (18, 38), 0.6: (12, 28), 0.3: (7, 20), 0.15: (5, 13), 0.075: (4, 8)},
    "Laston AC-Base": {37.5: (100, 100), 25: (90, 100), 19: (76, 90), 12.5: (60, 78), 9.5: (52, 71), 4.75: (35, 54), 2.36: (23, 41), 1.18: (13, 30), 0.6: (10, 22), 0.3: (6, 15), 0.15: (4, 10), 0.075: (3, 7)},
}
K_DEF = {"Latasir": 2.5, "Lataston": 2.5, "Laston": 0.75}
K_RNG = {"Latasir": (2.0, 3.0), "Lataston": (2.0, 3.0), "Laston": (0.5, 1.0)}

# Saringan hasil praktikum (mm) + contoh massa tertahan (gr), terakhir = pan
LAB = [76.2, 63.5, 50.8, 37.5, 19, 12.5, 9.5, 4.75, 2.36, 1.18, 0.6, 0.3, 0.15, 0.075]
SAMPLE = [0, 0, 0, 0, 0, 210, 883.6, 1206.4, 900.9, 502.7, 400.7, 274.7, 220.5, 200, 200.5]


# ------------------------------------------------ penyimpanan & riwayat akses
DATA_DIR = os.environ.get("GRADASPAL_DATA", "data")
ADMIN_CODE = os.environ.get("GRADASPAL_ADMIN", "nisa1536")
os.makedirs(os.path.join(DATA_DIR, "users"), exist_ok=True)
VISITS = os.path.join(DATA_DIR, "visits.csv")


def now_wib():
    return datetime.now(timezone(timedelta(hours=7))).strftime("%Y-%m-%d %H:%M:%S")


def user_path(uid):
    return os.path.join(DATA_DIR, "users", f"{uid}.json")


def load_user(uid):
    try:
        with open(user_path(uid), encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}


def save_user(uid, data):
    try:
        with open(user_path(uid), "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)
    except Exception:
        pass


def log_visit(name, event):
    try:
        ip = st.context.headers.get("X-Forwarded-For", "").split(",")[0].strip()
    except Exception:
        ip = ""
    new = not os.path.exists(VISITS)
    with open(VISITS, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["waktu (WIB)", "nama", "aktivitas", "ip"])
        w.writerow([now_wib(), name, event, ip])


def plot(mix, xs=None, ys=None, mids=False):
    sp = SPEC[mix]
    x = [s for s in S if s in sp]
    fig, ax = plt.subplots(figsize=(9, 5))
    fig.patch.set_facecolor("#FBF5EC")
    ax.set_facecolor("#FFFDF9")
    lo_v, hi_v = [sp[s][0] for s in x], [sp[s][1] for s in x]
    ax.fill_between(x, lo_v, hi_v, color="#E6CCB2", alpha=0.55, label="Daerah spesifikasi")
    ax.plot(x, lo_v, "o-", color="#5C3317", lw=2, label="BB")
    ax.plot(x, hi_v, "o-", color="#B08968", lw=2, label="BA")
    if mids:
        ax.plot(x, [sum(sp[s]) / 2 for s in x], "d--", color="#8B5A2B", lw=2, label="Nilai Tengah")
    if xs is not None:
        ax.plot(xs, ys, "o-", color="#E07B39", lw=2.5, label="Presentase Butir Lolos (%)")
    ax.set_xscale("log")
    ax.set_xlim(100, 0.01)
    ax.set_ylim(0, 120)
    ax.set_xticks([100, 10, 1, 0.1, 0.01])
    ax.set_xticklabels(["100", "10", "1", "0,1", "0,01"])
    ax.grid(True, which="major", alpha=0.4)
    ax.set_xlabel("Ukuran Lubang Saringan (mm)")
    ax.set_ylabel("Presentase Lolos Kumulatif (%)")
    ax.set_title("Hasil Perbandingan Presentase Lolos Kumulatif Terhadap Persen "
                 f"Berat Butir yang Lolos pada {mix}", fontsize=10, wrap=True)
    ax.legend(loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.28), frameon=False)
    fig.tight_layout()
    return fig


def show_fig(fig, name):
    st.pyplot(fig)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=200, bbox_inches="tight")
    st.download_button("Unduh grafik (PNG)", buf.getvalue(), f"{name}.png", "image/png", key=name)
    plt.close(fig)


st.set_page_config(page_title="GradAspal", page_icon="🛣️", layout="wide")
st.markdown("""
<style>
.stApp { background: linear-gradient(180deg, #FBF5EC 0%, #F1E3D0 100%); }
[data-testid="stSidebar"] { background: linear-gradient(180deg, #EAD7C0 0%, #D2B48C 100%); }
.hero { background: linear-gradient(135deg, #3E2415 0%, #6F4E37 55%, #A9744F 100%);
        padding: 1.6rem 2rem; border-radius: 16px; margin-bottom: 1.2rem;
        box-shadow: 0 6px 18px rgba(62,36,21,.28); }
.hero h1 { color: #FFF3E3; margin: 0; font-size: 2.3rem; letter-spacing: .5px; }
.hero p { color: #EBD5BA; margin: .3rem 0 0 0; font-size: 1rem; }
[data-testid="stMetric"] { background: rgba(255,255,255,.65); border-left: 6px solid #8B5A2B;
        border-radius: 12px; padding: .7rem 1rem; box-shadow: 0 2px 8px rgba(62,36,21,.12); }
.stTabs [data-baseweb="tab"] { font-weight: 600; }
.stDownloadButton button { background: linear-gradient(135deg, #6F4E37, #A9744F); color: #fff;
        border: none; border-radius: 10px; }
.stDownloadButton button:hover { background: linear-gradient(135deg, #5C3317, #8B5A2B); color: #fff; }
</style>
<div class="hero">
  <h1>🛣️ GradAspal</h1>
  <p>Perencanaan gradasi agregat dan kadar aspal estimasi untuk Latasir, Lataston, dan Laston</p>
</div>
""", unsafe_allow_html=True)

# ---------------------------------------------------------------- sidebar
name = st.sidebar.text_input("Nama pengguna", placeholder="Contoh: Budi / Kelompok 3")
pin = st.sidebar.text_input("Kode pribadi (PIN)", type="password",
                            help="Pertama kali: buat kode bebas dan ingat baik-baik. Selanjutnya: masukkan kode yang sama.")
st.sidebar.caption("Aktivitas akses dicatat. Pekerjaan tersimpan otomatis dan bisa dilanjutkan dengan nama dan kode yang sama.")
if not name.strip() or not pin:
    st.info("Masukkan **nama pengguna** dan **kode pribadi** di sidebar untuk mulai. Pemakaian pertama: buat kode sendiri. "
            "Kembali lagi: ketik nama dan kode yang sama agar pekerjaanmu dimuat.")
    st.stop()
uid = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_") or "pengguna"
pin_hash = hashlib.pbkdf2_hmac("sha256", pin.encode(), uid.encode(), 100_000).hex()
auth = f"{uid}:{pin_hash}"
if st.session_state.get("auth") != auth:
    saved = load_user(uid)
    if saved.get("pin_hash") and saved["pin_hash"] != pin_hash:
        if st.session_state.get("fail") != auth:
            log_visit(name.strip(), "kode salah")
            st.session_state["fail"] = auth
        st.error("Kode pribadi salah untuk nama ini. Coba lagi, atau minta admin mereset kode.")
        st.stop()
    st.session_state["auth"] = auth
    st.session_state["saved"] = saved
    log_visit(name.strip(), "lanjut" if saved else "baru")
sv = st.session_state.get("saved") or {}
ver = st.session_state.get("ver", 0)
tag = f"{uid}_{ver}"
if sv:
    st.sidebar.success(f"Data tersimpan dimuat (terakhir: {sv.get('diperbarui', '-')})")
if st.sidebar.button("Mulai baru (hapus data tersimpan)"):
    try:
        os.remove(user_path(uid))
    except OSError:
        pass
    st.session_state["saved"] = {}
    st.session_state["ver"] = ver + 1
    st.rerun()
mix_list = list(SPEC)
mix0 = sv.get("mix") if sv.get("mix") in SPEC else "Laston AC-WC"
mix = st.sidebar.selectbox("Jenis campuran", mix_list, index=mix_list.index(mix0), key=f"mix_{tag}")
W = st.sidebar.number_input("Berat campuran (gr)", 100.0, 10000.0, float(sv.get("W", 1200.0)), 50.0, key=f"W_{tag}")
fam = mix.split()[0]
k0 = float(sv["K"]) if sv.get("fam") == fam and "K" in sv else K_DEF[fam]
K = st.sidebar.number_input("Konstanta K", 0.0, 5.0, k0, 0.05, key=f"K_{tag}_{fam}")
lo_k, hi_k = K_RNG[fam]
st.sidebar.caption(f"Rentang K {fam}: {lo_k:g} - {hi_k:g}")
if not lo_k <= K <= hi_k:
    st.sidebar.warning("K di luar rentang yang lazim.")
sp = SPEC[mix]

tab1, tab2, tab3, tab4 = st.tabs(["1. Analisis Saringan", "2. Gradasi Rencana",
                                  "3. Kadar Aspal, Berat & Koreksi", "4. Riwayat Akses (Admin)"])

# ------------------------------------------------------------------ tab 1
with tab1:
    st.subheader("Input massa tertahan tiap saringan")
    base = pd.DataFrame({"Saringan (mm)": [f"{s:g}" for s in LAB] + ["Pan"],
                         "Massa Tertahan (gr)": sv["massa"] if len(sv.get("massa", [])) == len(LAB) + 1 else SAMPLE})
    ed = st.data_editor(base, hide_index=True, disabled=["Saringan (mm)"], width="stretch", key=f"massa_{tag}")
    m = ed["Massa Tertahan (gr)"].fillna(0).astype(float).values
    tot = m.sum()
    if tot <= 0:
        st.warning("Isi massa tertahan terlebih dahulu.")
        df1 = None
    else:
        cum = np.cumsum(m)
        pk = cum / tot * 100
        lw = 100 - pk
        mhb = pk[:-1].sum() / 100
        df1 = pd.DataFrame({"Saringan (mm)": list(base["Saringan (mm)"]),
                            "Massa Tertahan (gr)": m, "Jumlah Tertahan (gr)": cum,
                            "% Kumulatif Tertahan": pk, "% Lewat": lw})
        df1.loc[len(df1)] = ["Jumlah", tot, cum.sum(), pk[:-1].sum(), np.nan]
        st.dataframe(df1.round(2), hide_index=True, width="stretch")

        c1, c2 = st.columns(2)
        c1.metric("MHB", f"{mhb:.2f}")
        if 1.5 <= mhb <= 3.8:
            c2.success("MHB memenuhi syarat agregat halus/pasir (1,5 - 3,8)")
        elif 5.0 <= mhb <= 8.0:
            c2.success("MHB memenuhi syarat agregat kasar/kerikil (5,0 - 8,0)")
        else:
            c2.error("MHB di luar syarat pasir (1,5-3,8) dan kerikil (5,0-8,0)")

        rows = []
        for i, s in enumerate(LAB):
            if s in sp:
                lo, hi = sp[s]
                ok = lo - 0.005 <= lw[i] <= hi + 0.005
                rows.append([f"{s:g}", lo, hi, round(lw[i], 2), "Masuk batas" if ok else "Di luar batas"])
        if rows:
            st.subheader(f"Pengecekan terhadap batas {mix}")
            st.dataframe(pd.DataFrame(rows, columns=["Saringan (mm)", "BB", "BA", "% Lolos Sampel", "Status"]),
                         hide_index=True, width="stretch")
        show_fig(plot(mix, LAB, lw[:-1]), "grafik_saringan")

# ------------------------------------------------------------------ tab 2
mids, prev = [], 100.0
for s in S:
    if s in sp:
        prev = sum(sp[s]) / 2
    mids.append(prev)
mids = np.array(mids)
ret = np.r_[100.0, mids[:-1]] - mids  # saringan tanpa batas -> tertahan 0
with tab2:
    st.subheader(f"Gradasi rencana {mix} (nilai tengah)")
    df2 = pd.DataFrame({"Saringan (mm)": [f"{s:g}" for s in S],
                        "BB": [sp[s][0] if s in sp else np.nan for s in S],
                        "BA": [sp[s][1] if s in sp else np.nan for s in S],
                        "% Lolos (Nilai Tengah)": mids, "% Tertahan": ret})
    df2.loc[len(df2)] = ["Pan", np.nan, np.nan, 0.0, mids[-1]]
    df2.loc[len(df2)] = ["Jumlah", np.nan, np.nan, np.nan, ret.sum() + mids[-1]]
    st.dataframe(df2.round(2), hide_index=True, width="stretch")
    st.caption("Saringan tanpa batas di tabel spesifikasi mengikuti nilai tengah sebelumnya, sehingga % tertahannya 0.")
    show_fig(plot(mix, mids=True), "grafik_gradasi_rencana")

# ------------------------------------------------------------------ tab 3
with tab3:
    p236, p075 = mids[S.index(2.36)], mids[S.index(0.075)]
    AK, AH, F = 100 - p236, p236 - p075, p075
    Pb = 0.035 * AK + 0.045 * AH + 0.18 * F + K
    Pbr = float(np.floor(Pb * 2 + 0.5) / 2)

    st.subheader("Kadar aspal estimasi")
    a, b, c, d = st.columns(4)
    a.metric("AK (tertahan 2,36)", f"{AK:.2f}%")
    b.metric("AH (lolos 2,36 - lolos 0,075)", f"{AH:.2f}%")
    c.metric("F (lolos 0,075)", f"{F:.2f}%")
    d.metric("Pb estimasi", f"{Pb:.3f}%", f"dibulatkan {Pbr:g}%", delta_color="off")
    st.code(f"Pb = 0,035({AK:.2f}) + 0,045({AH:.2f}) + 0,18({F:.2f}) + {K:g} = {Pb:.3f}%  ->  {Pbr:g}%")

    kadar0 = float(sv["kadar"]) if sv.get("kadar_key") == f"{mix}|{K}" else Pbr
    kadar = st.number_input("Kadar aspal yang dipakai (%)", 0.0, 20.0, kadar0, 0.1, key=f"kadar_{tag}_{mix}_{K}")
    Wa = kadar / 100 * W
    Wg = W - Wa
    e, f_, g = st.columns(3)
    e.metric("Berat campuran", f"{W:.1f} gr")
    f_.metric("Berat aspal", f"{Wa:.2f} gr")
    g.metric("Berat agregat", f"{Wg:.2f} gr")

    st.subheader("Berat tertahan tiap saringan")
    rp = np.r_[ret, mids[-1]]
    df3 = pd.DataFrame({"Saringan (mm)": [f"{s:g}" for s in S] + ["Pan"],
                        "% Tertahan": rp, "Berat Tertahan (gr)": rp / 100 * Wg})
    df3.loc[len(df3)] = ["Jumlah", rp.sum(), rp.sum() / 100 * Wg]
    st.dataframe(df3.round(2), hide_index=True, width="stretch")

    st.subheader("Koreksi terhadap kadar aspal")
    fk = 1 - kadar / 100
    sebelum = [AK, AH, F, np.nan, AK + AH + F]
    koreksi = [AK * fk, AH * fk, F * fk, kadar]
    koreksi.append(sum(koreksi))
    df4 = pd.DataFrame({"Komponen": ["Agregat kasar (AK)", "Agregat halus (AH)", "Filler (F)", "Aspal", "Total"],
                        "Sebelum Koreksi (%)": sebelum, "Setelah Koreksi (%)": koreksi,
                        "Berat (gr)": [k / 100 * W for k in koreksi]})
    st.dataframe(df4.round(2), hide_index=True, width="stretch")
    st.caption("Koreksi: persentase agregat dikalikan (1 - kadar aspal), lalu ditambah aspal sehingga total 100%.")

    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as xw:
        if df1 is not None:
            df1.round(2).to_excel(xw, sheet_name="Analisis Saringan", index=False)
        df2.round(2).to_excel(xw, sheet_name="Gradasi Rencana", index=False)
        pd.DataFrame({"Parameter": ["Jenis campuran", "K", "AK", "AH", "F", "Pb estimasi", "Kadar dipakai",
                                    "Berat campuran", "Berat aspal", "Berat agregat"],
                      "Nilai": [mix, K, round(AK, 2), round(AH, 2), round(F, 2), round(Pb, 3), kadar,
                                W, round(Wa, 2), round(Wg, 2)]}
                     ).to_excel(xw, sheet_name="Kadar Aspal", index=False)
        df3.round(2).to_excel(xw, sheet_name="Berat per Saringan", index=False)
        df4.round(2).to_excel(xw, sheet_name="Koreksi", index=False)
    st.download_button("Unduh hasil (Excel)", buf.getvalue(), "hasil_gradaspal.xlsx",
                       "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet")

# ------------------------------------------- simpan otomatis & tab admin
save_user(uid, {"nama": name.strip(), "mix": mix, "fam": fam, "W": float(W), "K": float(K),
                "massa": [float(v) for v in m], "kadar": float(kadar), "kadar_key": f"{mix}|{K}",
                "diperbarui": now_wib(), "pin_hash": pin_hash})

with tab4:
    st.subheader("Riwayat akses pengguna")
    kode = st.text_input("Kode admin", type="password", key="admin_code")
    if kode and kode == ADMIN_CODE:
        if os.path.exists(VISITS):
            dv = pd.read_csv(VISITS)
            a1, a2 = st.columns(2)
            a1.metric("Total akses", len(dv))
            a2.metric("Pengguna unik", dv["nama"].str.lower().nunique())
            st.dataframe(dv.iloc[::-1], hide_index=True, width="stretch")
            st.download_button("Unduh riwayat (CSV)", dv.to_csv(index=False).encode("utf-8"),
                               "riwayat_akses.csv", "text/csv")
        else:
            st.info("Belum ada riwayat akses.")
        st.markdown("**Reset kode pengguna** (kalau ada yang lupa kode)")
        ulist = sorted(f[:-5] for f in os.listdir(os.path.join(DATA_DIR, "users")) if f.endswith(".json"))
        if ulist:
            target = st.selectbox("Pengguna", ulist, key="reset_user",
                                  format_func=lambda x: load_user(x).get("nama", x))
            if st.button("Reset kode pengguna ini"):
                d = load_user(target)
                d.pop("pin_hash", None)
                save_user(target, d)
                st.success("Kode direset. Pengguna bisa membuat kode baru saat masuk berikutnya.")
    elif kode:
        st.error("Kode admin salah.")
    else:
        st.caption("Tab ini hanya untuk pemilik aplikasi. Masukkan kode admin untuk melihat riwayat.")


Writing app.py


In [4]:
%%writefile .streamlit/config.toml
[theme]
base = "light"
primaryColor = "#8B5A2B"
backgroundColor = "#FBF5EC"
secondaryBackgroundColor = "#EAD7C0"
textColor = "#3E2415"
font = "sans serif"


Writing .streamlit/config.toml


In [3]:
# Simpan data & riwayat akses di Google Drive supaya tidak hilang saat sesi Colab berakhir
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['GRADASPAL_DATA'] = '/content/drive/MyDrive/GradAspal_data'
os.environ['GRADASPAL_ADMIN'] = 'nisa1536'   # GANTI dengan kode admin milikmu


Mounted at /content/drive


In [5]:
import subprocess, time, urllib.request

# jalankan GradAspal di latar belakang
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=open("log.txt", "w"), stderr=subprocess.STDOUT)
time.sleep(6)

ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip()
print("PASSWORD tunnel (kalau diminta):", ip)
print("Klik link 'your url is' di bawah ini, lalu isi password di atas jika muncul halaman Tunnel Password.")
!npx localtunnel --port 8501


PASSWORD tunnel (kalau diminta): 34.10.211.205
Klik link 'your url is' di bawah ini, lalu isi password di atas jika muncul halaman Tunnel Password.
⠙your url is: https://open-needles-speak.loca.lt
^C
